# Feature Engineering - ATP Tennis Match Predictor

This notebook builds the feature set used to predict ATP match winners based on findings from the EDA notebook ('01_eda.ipynb'). Rather than using raw match data directly, this notebook transforms it into features that describe the *relative* gap between two players: their ranking, points, and historical performance, since that's what actually determines who's more likely to win a given match.

**Key decisions carried over from EDA:**
- 'Pts_1'/'Pts_2' use '-1' as a missing data placeholder in ~23% of rows (not real NaN)
- 'Odd_1'/'Odd_2' are only reliably available from ~2005 onward
- 'rank_diff' (corr = -0.24) and 'points_diff' (corr = 0.32) both showed real relationships with match outcome and are treated as core features

In [65]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/processed/atp_matches_clean.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.shape

(68274, 21)

CSVs don't preserve datetime types, so 'Date' needs to be re-converted with 'pd.to_datetime()' every time the file is reloaded.

In [66]:
df['points_data_missing'] = ((df['Pts_1'] == -1) | (df['Pts_2'] == -1)).astype(int)

df['points_diff'] = np.where(
    df['points_data_missing'] == 1,
    0,
    df['Pts_1'] - df['Pts_2']
)

df['points_data_missing'].mean()

np.float64(0.22888654539063186)

'points_data_missing' is a separate binary column (1 = we don't know the points gap) that the model can learn to use as a signal, letting it learn to trust 'points_diff' less whenever this flag is on.

In [ ]:
df['has_odds'] = ((df['Odd_1'] != -1) & (df['Odd_2'] != -1)).astype(int)
df['has_odds'].mean()

np.float64(2.9293728212789642e-05)

'has_odds' is kept as a flag so any odds-based feature can later be built and evaluated only on the subset of rows where it's available

In [68]:
df = df.sort_values('Date').reset_index(drop=True)

The features in this notebook depends on chronological order. 'reset_index(drop=True)' renumbers rows after sorting

In [69]:
df = df.reset_index(drop=True)
df['match_id'] = df.index

p1 = df[['match_id', 'Date', 'Player_1', 'player_1_won', 'Surface']].copy()
p1.columns = ['match_id', 'Date', 'Player', 'Won', 'Surface']
p1['Slot'] = 'P1'

p2 = df[['match_id', 'Date', 'Player_2', 'player_1_won', 'Surface']].copy()
p2['Won'] = 1 - p2['player_1_won']
p2 = p2[['match_id', 'Date', 'Player_2', 'Won', 'Surface']]
p2.columns = ['match_id', 'Date', 'Player', 'Won', 'Surface']
p2['Slot'] = 'P2'

player_matches = pd.concat([p1, p2]).sort_values('Date').reset_index(drop=True)
print(df.shape[0] * 2 == player_matches.shape[0])
print(player_matches.duplicated(subset=['match_id', 'Player']).sum())
player_matches.head()

True
0


,match_id,Date,Player,Won,Surface,Slot
0,0,2000-01-03,Arthurs W.,0,Hard,P1
1,67,2000-01-03,Ilie A.,0,Hard,P2
2,66,2000-01-03,Fromberg R.,0,Hard,P2
3,65,2000-01-03,Henman T.,1,Hard,P2
4,64,2000-01-03,Henman T.,1,Hard,P2


### Why reshape the data?

Each row in 'df' is one match with two players side by side ('Player_1' and 'Player_2' as separate columns). To calculate something like "Federer's win rate going into this match," it's easier if each player's participation in a match is its own row, rather than being split across two columns depending on which side of the match they happened to be listed on.

In [70]:
player_matches['career_win_rate'] = (
    player_matches.groupby('Player')['Won']
    .transform(lambda x: x.shift().expanding().mean())
)

player_matches['surface_win_rate'] = (
    player_matches.groupby(['Player', 'Surface'])['Won']
    .transform(lambda x: x.shift().expanding().mean())
)

player_matches['recent_form'] = (
    player_matches.groupby('Player')['Won']
    .transform(lambda x: x.shift().rolling(window=10, min_periods=1).mean())
)

player_matches[['Player', 'Date', 'Won', 'career_win_rate', 'surface_win_rate', 'recent_form']].tail()

,Player,Date,Won,career_win_rate,surface_win_rate,recent_form
136543,Rublev A.,2026-07-18,1,0.630931,0.662162,0.6
136544,Collignon R.,2026-07-19,0,0.513514,0.615385,0.6
136545,Rublev A.,2026-07-19,1,0.631579,0.664430,0.6
136546,Darderi L.,2026-07-19,0,0.545455,0.693182,0.6
136547,Tsitsipas S.,2026-07-19,1,0.650000,0.728916,0.6


### Leakage-safe rolling statistics

If a player's win rate for a given match accidentally includes the result of that same match, or a match that hasn't happened yet, the model is effectively being shown the answer before making its prediction.

In [71]:
p1_stats = player_matches[player_matches['Slot'] == 'P1'].rename(columns={
    'career_win_rate': 'p1_career_win_rate',
    'surface_win_rate': 'p1_surface_win_rate',
    'recent_form': 'p1_recent_form'
})[['match_id', 'p1_career_win_rate', 'p1_surface_win_rate', 'p1_recent_form']]

p2_stats = player_matches[player_matches['Slot'] == 'P2'].rename(columns={
    'career_win_rate': 'p2_career_win_rate',
    'surface_win_rate': 'p2_surface_win_rate',
    'recent_form': 'p2_recent_form'
})[['match_id', 'p2_career_win_rate', 'p2_surface_win_rate', 'p2_recent_form']]

rows_before = df.shape[0]
df = df.merge(p1_stats, on=['match_id'], how='left')
df = df.merge(p2_stats, on=['match_id'], how='left')
print(rows_before == df.shape[0])

True


### Debugging note: merge row-count mismatch

Initial merge on '['Date', 'Player']' caused row count to increase from 68,274 to 125,629. Investigation showed the raw data has zero true duplicate matches, the issue was that players can legitimately play multiple real matches on the same calendar date (compressed tournament draws, and Masters Cup round-robin groups against different opponents). Adding 'Round' to the merge key fixed most cases but still failed on round-robin groups, since 'Round' is the same label ("Round Robin") for multiple distinct matches in a group stage.

Fixed by assigning each match row a unique 'match_id' before reshaping, and merging on that instead of trying to find a naturally unique combination of columns.